# Embedding distance vs LLM judges for research-idea novelty

**Question:** on Ai2's new novelty-judge benchmark, how close do established embedding novelty metrics (no LLM at all) get to an LLM judge that uses Ai2's own prompt, and to the frontier-judge accuracies Ai2 reports?

**Setup**
- **Benchmark:** [`allenai/novelty-judge-bench`](https://huggingface.co/datasets/allenai/novelty-judge-bench) (ODC-BY). Ideas are ICLR 2026 submissions whose reviewers explicitly praised (154) or faulted (145) their originality, plus 154 ideas from a plain LLM generator. **Human-Only** is the main split; **Human+Generated** is secondary, because there the lower-novelty side is LLM-written and distance could pick up writing style.
- **Prior work:** 576k ML papers from the [Papers with Code archive](https://huggingface.co/datasets/pwc-archive/papers-with-abstracts) (CC BY-SA 4.0), which stops in July 2025, before the ICLR 2026 deadline.
- **Conditions:**
  1. **Yin:** minimum embedding distance to prior work (Yin et al., 2023)
  2. **kNN density:** mean distance to the 10 nearest papers
  3. **RND:** relative neighbour density (Wang et al., 2025)
  4. **LLM judge:** Qwen2.5-3B-Instruct (released before the ICLR 2026 submissions existed) on Ai2's reference prompt and two of its prompt ablations
  5. **Yin + judge:** logistic regression over both, cross-validated
- **Metrics:** pairwise soft and strict accuracy and pointwise macro-F1 (Ai2's metrics), AUROC, seconds per 100 decisions, and a check for whether each score tracks reviewer *rating* rather than novelty.

**TL;DR:** TODO(you) after running.

## Why this matters

Ai2's [Old Ideas, Novel Problems](https://arxiv.org/abs/2610.02022) (October 2026) showed that LLM novelty judges are unstable: rewording the prompt moves pairwise accuracy by more than 50 points, retrieval barely helps, and two purpose-built novelty systems lose to a plain prompted baseline. The study cost about $15,000 in API calls, and every condition in it is an LLM.

A separate line of work scores novelty without an LLM, as distance from prior work in embedding space. Liu & Zhai's [axiomatic benchmark](https://arxiv.org/abs/2604.15145) (2026) found the simplest of these, Yin et al.'s minimum distance, the strongest of ten systems on their probes, but it has never been run on Ai2's review-mined labels. This notebook puts the two side by side on the same items. The embedding metrics aren't new; the comparison is.

## Setup

Runs on a free Colab T4 in about 30–40 minutes: about 10 minutes to download and embed the corpus once (cached), and 10–20 minutes for the judge. No tokens or paid APIs are needed; everything is public.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

%pip install -q "transformers==5.22.1" "sentence-transformers==6.2.0" "datasets==5.1.0" "accelerate==1.15.0"

In [ ]:
import ast, gc, hashlib, json, math, re, shutil, time, urllib.request
from pathlib import Path

import jinja2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from datasets import load_dataset
from huggingface_hub import snapshot_download
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from importlib.metadata import version
print({p: version(p) for p in ["transformers", "sentence-transformers", "datasets", "torch", "scikit-learn", "pandas"]})
print("Python", sys.version.split()[0], "| Colab" if IN_COLAB else "| local")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Config

In [ ]:
SMOKE = False                     # True: 30k-paper corpus, 20 pairs and 40 pointwise ideas per split, separate cache (about 10 minutes)
USE_DRIVE = True                  # cache the corpus embeddings and judge outputs on Google Drive so a disconnect doesn't lose them

# Benchmark and corpus, pinned to the Hub revisions checked on 1 November 2026
BENCH_ID, BENCH_REV = "allenai/novelty-judge-bench", "fe201e9638efbf8562413de2d3f50f9b661910c4"
CORPUS_ID, CORPUS_REV = "pwc-archive/papers-with-abstracts", "459480aeac58509a93f5af2acd4adf20236ce8d9"
CORPUS_CUTOFF = "2025-09-19"      # ICLR 2026 abstract deadline. The archive stops in July 2025, so this is only a guard
SPLITS = {"Human-Only": "human-only", "Human+Generated": "human-plus-generated"}

# Ai2's judge prompts and criteria are read from their repo (Apache-2.0) at this commit
AI2_COMMIT = "9b2e69f3dfb46f30ffaa38414e581ad38322bf2b"

# Embedding models, both trained long before the ICLR 2026 submissions: (Hub id, revision, max tokens)
EMBEDDERS = {
    "bge-small": ("BAAI/bge-small-en-v1.5", "5c38ec7c405ec4b44b94cc5a9bb96e735b38267a", 256),
    "bge-m3": ("BAAI/bge-m3", "5617a9f61b028005a4858fdac845db406aefb181", 512),  # Liu & Zhai's choice; much slower on the corpus
}
EMBEDDER = "bge-small"
EMBED_BATCH = 512

# Embedding novelty metrics
KNN_K = 10                        # kNN density: mean distance to the 10 nearest papers
RND_P, RND_Q = 100, 50            # RND: P neighbours, Q papers per density estimate (the paper's values)
LEAK_COS = 0.95                   # a corpus paper at least this similar to an idea is treated as the idea itself and dropped
MARGIN = 50                       # extra neighbours fetched so P remain after the leakage filter

# LLM judges: name -> (Hub id, revision, trust_remote_code)
JUDGES = {
    "Qwen2.5-3B": ("Qwen/Qwen2.5-3B-Instruct", "aa8e72537993ba99e69dfaafa59ed015b17504d1", False),  # September 2024
}
RUN_LFM3 = False                  # True: add LFM3-1.7B (October 2026) as a possibly contaminated contrast judge
if RUN_LFM3:
    JUDGES["LFM3-1.7B"] = ("LiquidAI/LFM3-1.7B", "ac2586a34f1a1c03329dbdcdefabd8b6478ddaa7", True)
JUDGE_DTYPE = "float16"           # T4 has no fast bfloat16. Set "float32" if the judge returns NaN probabilities (12 GB for 3B)
PROMPT_VARIANTS = ["reference", "vague_criteria", "no_criteria"]  # Ai2's reference prompt and its P1/P2 ablations
RUN_RETRIEVAL_JUDGE = False       # True: also run the reference prompt with the 5 nearest corpus abstracts as related work
RETRIEVAL_K = 5

# Frontier-judge pairwise soft accuracies read off Ai2's Figure 4 (reference configuration), for the comparison chart
AI2_REFERENCE = {
    # "Human-Only": {"claude-opus-4-6": 0.00, "gpt-5.4": 0.00},   # TODO(you): fill from the paper
}

N_SMOKE_CORPUS, N_SMOKE_ITEMS = 30_000, 20
SLUG = "2026-11-novelty-embedding-baseline"

In [ ]:
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/ML") / SLUG
else:
    CACHE = Path("outputs")
if SMOKE:
    CACHE = CACHE / "smoke"
CACHE.mkdir(parents=True, exist_ok=True)
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
OUT = Path("results"); OUT.mkdir(exist_ok=True)
print("cache:", CACHE.resolve())

## Data

### Benchmark

Each idea is an abstract. For human ideas, Ai2 had Claude Opus 4.6 strip numerical results and claims of beating specific baselines, so the text is close to, but not the same as, the submission's abstract. Pairs always share an ICLR area. Licence: **ODC-BY**.

In [ ]:
def norm_title(t):
    return re.sub(r"[^a-z0-9]", "", str(t).lower())


def text_key(t):
    return hashlib.sha1(t.strip().encode()).hexdigest()[:16]


pairs, points = {}, {}
for split, prefix in SPLITS.items():
    pw = load_dataset(BENCH_ID, f"{prefix}_pairwise", split="test", revision=BENCH_REV).to_pandas()
    pt = load_dataset(BENCH_ID, f"{prefix}_pointwise", split="test", revision=BENCH_REV).to_pandas()
    if SMOKE:
        pw = pw.sample(N_SMOKE_ITEMS, random_state=SEED)
        pt = pt.groupby("label", group_keys=False).sample(N_SMOKE_ITEMS, random_state=SEED)
    pt["y"] = (pt.label == "POSITIVE").astype(int)
    pairs[split], points[split] = pw.reset_index(drop=True), pt.reset_index(drop=True)
    print(f"{split}: {len(pw)} pairs (novel idea is B in {int(pw.expected_winner.sum())}), "
          f"{len(pt)} pointwise ideas {pt.label.value_counts().to_dict()}")

# One row per unique idea text, across both splits and formats
rows = {}
for split in SPLITS:
    for r in points[split].itertuples():
        rows.setdefault(text_key(r.idea), dict(idea=r.idea, title=r.title, forum_id=r.forum_id, source=r.idea_source,
                                               label=r.label, area=r.iclr_area, rating=r.rating))
    for r in pairs[split].itertuples():
        for s in "ab":
            t = getattr(r, f"idea_{s}")
            rows.setdefault(text_key(t), dict(idea=t, title=getattr(r, f"title_{s}"), forum_id=getattr(r, f"forum_id_{s}"),
                                              source=getattr(r, f"idea_source_{s}"), label=getattr(r, f"label_{s}"),
                                              area=r.iclr_area, rating=getattr(r, f"rating_{s}")))
ideas = pd.DataFrame.from_dict(rows, orient="index").rename_axis("key").reset_index()
key_pos = {k: n for n, k in enumerate(ideas.key)}
print(f"{len(ideas)} unique ideas:", ideas.groupby(["source", "label"]).size().to_dict())

for r in points["Human-Only"].groupby("label").head(2).head(3).itertuples():
    print(f"\n[{r.label}] {r.title} (rating {r.rating})\n{r.idea[:400]}...")

### Prior-work corpus

The [Papers with Code archive](https://huggingface.co/datasets/pwc-archive/papers-with-abstracts) (licence **CC BY-SA 4.0**) is a snapshot of 576k ML papers with arXiv abstracts and dates, frozen when the site shut down in July 2025. Liu & Zhai built their pools from the same archive. Every paper predates the ICLR 2026 deadline, which is the same restriction Ai2 applied to its retrieval. Only titles and abstracts are used; papers without a usable abstract or date, and duplicate titles, are dropped.

In [ ]:
corpus_path = CACHE / "corpus.parquet"
if corpus_path.exists():
    corpus = pd.read_parquet(corpus_path)
else:
    local = snapshot_download(CORPUS_ID, repo_type="dataset", revision=CORPUS_REV, allow_patterns=["data/*.parquet"])
    corpus = pd.read_parquet(Path(local) / "data", columns=["arxiv_id", "openreview_id", "title", "abstract", "date"])
    n0 = len(corpus)
    corpus = corpus[corpus.abstract.fillna("").str.len() > 200]
    corpus = corpus[corpus.date.notna() & (corpus.date < pd.Timestamp(CORPUS_CUTOFF))]
    corpus["tnorm"] = corpus.title.map(norm_title)
    corpus = corpus.drop_duplicates("tnorm").reset_index(drop=True)
    print(f"{n0:,} papers -> {len(corpus):,} with an abstract, a date before {CORPUS_CUTOFF} and a unique title")
    corpus.to_parquet(corpus_path)
if SMOKE and len(corpus) > N_SMOKE_CORPUS:
    corpus = corpus.sample(N_SMOKE_CORPUS, random_state=SEED).reset_index(drop=True)
print(f"corpus: {len(corpus):,} papers, {corpus.date.min():%Y-%m} to {corpus.date.max():%Y-%m}")

## Embedding novelty metrics (conditions 1–3)

Abstracts on both sides are embedded with the same model and compared by cosine similarity (embeddings are unit length, so a dot product). Then, for each idea:

- **Leakage filter.** Many ICLR submissions were on arXiv before the deadline, so an idea's own preprint can be in the corpus. A corpus paper is dropped from an idea's neighbours if its OpenReview ID or normalised title matches the idea, or if its cosine similarity is at least `LEAK_COS`. Every dropped neighbour is logged for a spot check.
- **Yin:** `1 − max cosine` to the remaining corpus.
- **kNN density:** `1 − mean cosine` over the `KNN_K` nearest.
- **RND** (Wang et al., 2025): the idea's neighbour density `ND` is its mean cosine distance to its `Q` nearest papers. The same is computed for each of its `P` nearest papers, using their own nearest papers. The score is the percentile of the idea's `ND` among its neighbours' (0–100): an idea in a sparser spot than its neighbours scores high.

A higher score means more novel for all three. They are deterministic, so ties are the only way a pair can be undecided.

In [ ]:
emb_name, emb_rev, emb_len = EMBEDDERS[EMBEDDER]
embedder = SentenceTransformer(emb_name, revision=emb_rev, device=DEVICE)
embedder.max_seq_length = emb_len
if DEVICE == "cuda":
    embedder.half()


def embed(texts):
    return embedder.encode(list(texts), batch_size=EMBED_BATCH, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=len(texts) > 1000).astype(np.float16)


emb_path, timing_path = CACHE / f"corpus_emb_{EMBEDDER}.npy", CACHE / f"timing_{EMBEDDER}.json"
if emb_path.exists() and len(np.load(emb_path, mmap_mode="r")) == len(corpus):
    C = np.load(emb_path)
    corpus_embed_s = json.loads(timing_path.read_text())["corpus_embed_s"] if timing_path.exists() else float("nan")
else:
    t0 = time.time()
    C = embed(corpus.abstract)
    corpus_embed_s = time.time() - t0
    np.save(emb_path, C)
    timing_path.write_text(json.dumps({"corpus_embed_s": corpus_embed_s, "n": len(corpus)}))

t0 = time.time()
Q = embed(ideas.idea)
idea_embed_s = time.time() - t0
print(f"corpus {C.shape}: {corpus_embed_s / 60:.1f} min to embed (one-off); {len(Q)} ideas: {idea_embed_s:.1f} s")

In [ ]:
Ct = torch.from_numpy(C).to(DEVICE)
if DEVICE == "cpu":
    Ct = Ct.float()


def topk_search(Qnp, k, batch=1024):
    """Top-k cosine neighbours in the corpus. Returns (similarities, corpus row indices)."""
    sims, idx = [], []
    for i in range(0, len(Qnp), batch):
        q = torch.from_numpy(np.asarray(Qnp[i:i + batch])).to(DEVICE, Ct.dtype)
        s, j = (q @ Ct.T).topk(k, dim=1)
        sims.append(s.float().cpu().numpy()); idx.append(j.cpu().numpy())
    return np.concatenate(sims), np.concatenate(idx)


t0 = time.time()
S, I = topk_search(Q, RND_P + MARGIN)

# Leakage filter
or_map = {k: set(v) for k, v in corpus.dropna(subset=["openreview_id"]).groupby("openreview_id").groups.items()}
t_map = dict(zip(corpus.tnorm, corpus.index))
keep_s, keep_i, leaks = [], [], []
for n, r in enumerate(ideas.itertuples()):
    banned = set(or_map.get(r.forum_id, ())) if isinstance(r.forum_id, str) else set()
    if r.source == "human" and t_map.get(norm_title(r.title)) is not None:
        banned.add(t_map[norm_title(r.title)])
    ok = []
    for s, j in zip(S[n], I[n]):
        if j in banned or s >= LEAK_COS:
            leaks.append(dict(key=r.key, idea_title=r.title, corpus_title=corpus.title[j], cos=round(float(s), 3),
                              reason="id/title match" if j in banned else f"cos >= {LEAK_COS}"))
        else:
            ok.append((s, j))
    if len(ok) < RND_P:
        raise RuntimeError(f"idea {r.key}: only {len(ok)} neighbours left after the leakage filter; raise MARGIN")
    keep_s.append([s for s, _ in ok[:RND_P]]); keep_i.append([j for _, j in ok[:RND_P]])
NS, NI = np.array(keep_s), np.array(keep_i)
leaks = pd.DataFrame(leaks, columns=["key", "idea_title", "corpus_title", "cos", "reason"])

# Scores
ideas["yin"] = 1 - NS[:, 0]
ideas["knn"] = 1 - NS[:, :KNN_K].mean(1)
nd_idea = 1 - NS[:, :RND_Q].mean(1)
uniq = np.unique(NI)
s_n, _ = topk_search(C[uniq], RND_Q + 1)           # column 0 is the paper itself
nd_all = np.full(len(corpus), np.nan)
nd_all[uniq] = 1 - s_n[:, 1:].mean(1)
ideas["rnd"] = 100 * (nd_all[NI] <= nd_idea[:, None]).mean(1)
ideas["nn_title"], ideas["nn_cos"] = corpus.title.values[NI[:, 0]], NS[:, 0]
search_s = time.time() - t0
EMB_SEC_PER_100 = (idea_embed_s + search_s) / len(ideas) * 100
METRICS = {"yin": "Yin (min distance)", "knn": f"kNN density (k={KNN_K})", "rnd": "RND"}

print(f"search + filter + scores: {search_s:.1f} s for {len(ideas)} ideas ({EMB_SEC_PER_100:.2f} s per 100 ideas incl. embedding)")
print(f"leakage filter dropped {len(leaks)} neighbours from {leaks.key.nunique()} ideas:", leaks.reason.value_counts().to_dict())
ideas[["title", "source", "label", "yin", "knn", "rnd", "nn_cos", "nn_title"]].head()

**Spot check the leakage filter.** The rows below should be an idea's own preprint (same paper, maybe retitled) or a near-copy. If clearly different papers show up, raise `LEAK_COS`; if obvious preprints slip through, check the nearest-neighbour similarities in the qualitative section and lower it.

In [ ]:
with pd.option_context("display.max_colwidth", 70):
    display(leaks.sort_values("cos").head(10))

## LLM judge (condition 4)

The judge gets **Ai2's own prompt and novelty criteria**, rendered from their repo at a pinned commit, in three versions:

- **reference:** Ai2's reference configuration, including the pairwise "Tip" line
- **vague_criteria:** Ai2's P1 ablation, with the guardrail sentences removed from the criteria
- **no_criteria:** Ai2's P2 ablation, with no criteria at all

**Two deviations from Ai2's setup:**

- **Log-probabilities, not generated reasoning.** Ai2's judges write a `<thinking>` block and then a JSON verdict, and each decision aggregates 3 samples. Here the reply is prefilled up to `{"novelty": ` with an empty thinking block, and the verdict is read from the probabilities of the next token being `0` or `1`. That's one deterministic forward pass per prompt, and it gives a probability you can threshold, average or feed into a classifier. It also means the small judge never gets to reason, so read its numbers as a lower bound.
- **No sampling:** each pair runs once per presentation order (A then B, and B then A), and the two probabilities are averaged, so position bias cancels out in the verdict. How often the two orders disagree is reported as the **order-flip rate**.

**Contamination.** Ai2 chose judges whose training data predates the ICLR 2026 deadline. Qwen2.5-3B-Instruct (September 2024) meets that bar. LFM3-1.7B (October 2026, optional) may have seen these papers and their reviews, so it's only a contrast.

In [ ]:
AI2 = f"https://raw.githubusercontent.com/allenai/novelty-eval/{AI2_COMMIT}/src/novelty_eval/"


def fetch(path):
    with urllib.request.urlopen(AI2 + path) as f:
        return f.read().decode()


_names = {"EVALUATION_CRITERIA", "EVALUATION_CRITERIA_RETRIEVAL", "NOVELTY_PAIRWISE_TIP", "NOVELTY_RETRIEVAL_PAIRWISE_TIP"}
AI2C = {node.targets[0].id: ast.literal_eval(node.value) for node in ast.parse(fetch("judge.py")).body
        if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name) and node.targets[0].id in _names}
VAGUE = yaml.safe_load(fetch("ablation/ablations.yaml"))["shared"]["vague_novelty_criterion"]
TEMPLATES = {n: jinja2.Template(fetch(f"templates/{n}.jinja2")) for n in
             ["pairwise_novelty", "pairwise_novelty_no_criteria", "pointwise_novelty", "pointwise_novelty_no_criteria"]}
assert _names <= set(AI2C), set(AI2C)


def with_tip(criteria, tip):
    """Ai2's get_pairwise_novelty_prompt: append the tip to the novelty criterion at the same indent."""
    text = criteria["novelty"]
    first = next(l for l in text.splitlines() if l.strip())
    return {"novelty": text.rstrip() + "\n\n" + " " * (len(first) - len(first.lstrip())) + tip}


def build_prompt(fmt, variant, texts, related=None):
    retrieval = variant == "retrieval"
    crit = AI2C["EVALUATION_CRITERIA_RETRIEVAL" if retrieval else "EVALUATION_CRITERIA"]
    if variant == "vague_criteria":
        crit = {"novelty": VAGUE}
    tmpl = TEMPLATES[f"{fmt}_novelty" + ("_no_criteria" if variant == "no_criteria" else "")]
    related = related or [""] * len(texts)
    if fmt == "pointwise":
        return tmpl.render(idea={"text": texts[0], "related_work": related[0]}, evaluation_criteria=crit)
    if variant != "no_criteria":
        crit = with_tip(crit, AI2C["NOVELTY_RETRIEVAL_PAIRWISE_TIP" if retrieval else "NOVELTY_PAIRWISE_TIP"])
    return tmpl.render(idea0={"text": texts[0], "related_work": related[0]},
                       idea1={"text": texts[1], "related_work": related[1]}, evaluation_criteria=crit)


def related_for(key):
    return "\n\n".join(f"[{i + 1}] {corpus.title[j]}: {corpus.abstract[j][:1500]}"
                       for i, j in enumerate(NI[key_pos[key], :RETRIEVAL_K]))


p = points["Human-Only"]
print(build_prompt("pairwise", "reference", [p.idea[0][:300] + " ...", p.idea[1][:300] + " ..."]))

In [ ]:
PREFILL = '<thinking>\n</thinking>\n\n```json\n{"novelty": '


def load_judge(name):
    mid, rev, remote = JUDGES[name]
    tok = AutoTokenizer.from_pretrained(mid, revision=rev, trust_remote_code=remote)
    dtype = getattr(torch, JUDGE_DTYPE) if DEVICE == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(mid, revision=rev, dtype=dtype, device_map=DEVICE,
                                                 trust_remote_code=remote).eval()
    digits = [tok.encode(d, add_special_tokens=False) for d in "01"]
    assert all(len(d) == 1 for d in digits), f"'0'/'1' are not single tokens for {name}: {digits}"
    return tok, model, [d[0] for d in digits]


@torch.no_grad()
def p_one(tok, model, digits, prompt):
    """Probability that the next token after the prefilled JSON key is '1' rather than '0'."""
    text = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    enc = tok(text + PREFILL, return_tensors="pt", add_special_tokens=False).to(model.device)
    logits = model(**enc).logits[0, -1, digits].float()
    return float(torch.softmax(logits, -1)[1]), int(enc.input_ids.shape[1])


VARIANTS = PROMPT_VARIANTS + (["retrieval"] if RUN_RETRIEVAL_JUDGE else [])
point_keys = sorted({text_key(t) for split in SPLITS for t in points[split].idea})
idea_text = dict(zip(ideas.key, ideas.idea))

# The judge doesn't need the corpus embeddings; free GPU memory first
Ct = None; gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

J = {}
for name in JUDGES:
    cache_file = CACHE / f"judge_{name}.jsonl"
    done = {}
    if cache_file.exists():
        for line in cache_file.read_text().splitlines():
            d = json.loads(line); done[d["id"]] = d
    jobs = {}
    for variant in VARIANTS:
        for split in SPLITS:
            for r in pairs[split].itertuples():
                ka, kb = text_key(r.idea_a), text_key(r.idea_b)
                jobs[f"{name}|{variant}|pair|{ka}|{kb}"] = ("pairwise", variant, [ka, kb])
                jobs[f"{name}|{variant}|pair|{kb}|{ka}"] = ("pairwise", variant, [kb, ka])
        for k in point_keys:
            jobs[f"{name}|{variant}|point|{k}"] = ("pointwise", variant, [k])
    todo = [j for j in jobs if j not in done]
    print(f"{name}: {len(jobs)} prompts, {len(todo)} still to run")
    if todo:
        tok, model, digits = load_judge(name)
        with open(cache_file, "a") as f:
            for jid in tqdm(todo, desc=name):
                fmt, variant, keys = jobs[jid]
                related = [related_for(k) for k in keys] if variant == "retrieval" else None
                t0 = time.time()
                p1, ntok = p_one(tok, model, digits, build_prompt(fmt, variant, [idea_text[k] for k in keys], related))
                if not math.isfinite(p1):
                    raise RuntimeError(f"{name} returned NaN on {jid}; set JUDGE_DTYPE = 'float32' and rerun")
                done[jid] = dict(id=jid, p1=p1, tokens=ntok, seconds=time.time() - t0)
                f.write(json.dumps(done[jid]) + "\n"); f.flush()
        del model; gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    J.update({j: done[j] for j in jobs})
print(f"{len(J)} judge decisions; mean {np.mean([d['tokens'] for d in J.values()]):.0f} prompt tokens, "
      f"{np.mean([d['seconds'] for d in J.values()]):.2f} s each")

## Evaluation

Ai2's metrics, applied to every condition in the same way:

- **Pairwise soft accuracy** (ties earn half credit) and **strict accuracy** (ties earn nothing). A score-based condition picks the idea with the higher score.
- **Pointwise macro-F1.** The embedding scores have no natural cut-off, so each condition gets a threshold chosen on the other folds of a 5-fold split (**CV F1**). The judge also gets the plain 0.5 cut-off (**F1@0.5**), which is how Ai2 used its judges.
- **AUROC:** how well the score separates novel from lower-novelty ideas, at any threshold.
- **Yin + judge:** a logistic regression on the Yin score and the judge's reference-prompt probability, with 5-fold cross-validated predictions. For pairs, the features are the differences between the two ideas.

In [ ]:
SC = ideas.set_index("key")


def pair_eval(score_a, score_b, winner):
    d = np.sign(np.asarray(score_b, float) - np.asarray(score_a, float))   # +1: B wins, -1: A wins, 0: tie
    truth = np.where(np.asarray(winner) == 1, 1, -1)
    correct, ties = d == truth, d == 0
    return dict(pair_soft=float((correct + 0.5 * ties).mean()), pair_strict=float(correct.mean()), pair_ties=int(ties.sum()))


def cv_f1(score, y, folds=5):
    score, y = np.asarray(score, float), np.asarray(y)
    pred = np.zeros_like(y)
    for tr, te in StratifiedKFold(folds, shuffle=True, random_state=SEED).split(score, y):
        best = max(np.unique(score[tr]), key=lambda t: f1_score(y[tr], score[tr] >= t, average="macro"))
        pred[te] = score[te] >= best
    return float(f1_score(y, pred, average="macro"))


def cv_logreg(X, y, folds=5):
    """Out-of-fold probabilities from a standardised logistic regression."""
    X, y = np.asarray(X, float), np.asarray(y)
    X = (X - X.mean(0)) / (X.std(0) + 1e-9)
    out = np.zeros(len(y))
    for tr, te in StratifiedKFold(folds, shuffle=True, random_state=SEED).split(X, y):
        out[te] = LogisticRegression().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return out


def judge_pair_probs(name, variant, pw):
    """P(B is more novel), averaged over both presentation orders, and whether the two orders disagree."""
    ka, kb = pw.idea_a.map(text_key), pw.idea_b.map(text_key)
    p_ab = np.array([J[f"{name}|{variant}|pair|{a}|{b}"]["p1"] for a, b in zip(ka, kb)])       # P(second idea, B)
    p_ba = np.array([J[f"{name}|{variant}|pair|{b}|{a}"]["p1"] for a, b in zip(ka, kb)])       # P(second idea, A)
    return (p_ab + 1 - p_ba) / 2, (p_ab > 0.5) != (p_ba < 0.5)


def judge_point_probs(name, variant, pt):
    return np.array([J[f"{name}|{variant}|point|{text_key(t)}"]["p1"] for t in pt.idea])


def sec_per_100(name, variant, kind):
    s = [d["seconds"] for k, d in J.items() if k.startswith(f"{name}|{variant}|{kind}|")]
    return float(np.mean(s)) * (200 if kind == "pair" else 100)    # a pair costs two prompts


rows = []
for split in SPLITS:
    pw, pt = pairs[split], points[split]
    ka, kb, kp = pw.idea_a.map(text_key), pw.idea_b.map(text_key), pt.idea.map(text_key)
    for m, label in METRICS.items():
        sp = SC.loc[kp, m].values
        rows.append(dict(split=split, condition=label, **pair_eval(SC.loc[ka, m].values, SC.loc[kb, m].values, pw.expected_winner),
                         point_f1_cv=cv_f1(sp, pt.y), point_auroc=roc_auc_score(pt.y, sp), sec_per_100=EMB_SEC_PER_100))
    for name in JUDGES:
        for variant in VARIANTS:
            pb, flips = judge_pair_probs(name, variant, pw)
            pp = judge_point_probs(name, variant, pt)
            rows.append(dict(split=split, condition=f"{name} judge ({variant})", **pair_eval(1 - pb, pb, pw.expected_winner),
                             order_flip=float(flips.mean()), point_f1_cv=cv_f1(pp, pt.y),
                             point_f1_05=float(f1_score(pt.y, pp >= 0.5, average="macro")), point_auroc=roc_auc_score(pt.y, pp),
                             sec_per_100=sec_per_100(name, variant, "point")))
        # Yin + judge (reference prompt)
        pb, _ = judge_pair_probs(name, "reference", pw)
        Xp = np.c_[SC.loc[kb, "yin"].values - SC.loc[ka, "yin"].values, pb - 0.5]
        pc = cv_logreg(Xp, pw.expected_winner)
        pp = cv_logreg(np.c_[SC.loc[kp, "yin"].values, judge_point_probs(name, "reference", pt)], pt.y)
        rows.append(dict(split=split, condition=f"Yin + {name} judge", **pair_eval(1 - pc, pc, pw.expected_winner),
                         point_f1_cv=float(f1_score(pt.y, pp >= 0.5, average="macro")), point_auroc=roc_auc_score(pt.y, pp)))

summary = pd.DataFrame(rows)
summary.to_csv(OUT / "summary.csv", index=False)
with pd.option_context("display.float_format", "{:.3f}".format):
    display(summary)

### Does the score track novelty, or just quality?

Ai2 drew the two pools from opposite ends of the ICLR rating distribution, and Liu & Zhai found reviewers' novelty and quality judgments entangled. So a score could separate the pools because it tracks how *good* a paper is, not how *new* it is. Within a single pool, ratings vary but every idea has the same novelty label, so a strong correlation with rating inside a pool points to quality rather than novelty. Human ideas from Human-Only only.

In [ ]:
pt = points["Human-Only"]
pt = pt[pt.idea_source == "human"]
cols = {label: SC.loc[pt.idea.map(text_key), m].values for m, label in METRICS.items()}
for name in JUDGES:
    cols[f"{name} judge (reference)"] = judge_point_probs(name, "reference", pt)
conf = []
for label, v in cols.items():
    for pool, mask in [("novel pool", pt.y == 1), ("lower-novelty pool", pt.y == 0), ("both pools", pt.y >= 0)]:
        rho, pval = spearmanr(v[mask], pt.rating[mask])
        conf.append(dict(score=label, pool=pool, spearman_rho=rho, p=pval, n=int(mask.sum())))
confound = pd.DataFrame(conf)
confound.to_csv(OUT / "quality_confound.csv", index=False)
with pd.option_context("display.float_format", "{:.3f}".format):
    display(confound.pivot(index="score", columns="pool", values="spearman_rho"))

## Results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
plot = summary.pivot(index="condition", columns="split", values="pair_soft").reindex(summary.condition.unique())
plot.plot.barh(ax=ax, width=0.8)
ax.axvline(0.5, color="grey", ls="--", lw=1, label="chance")
for split, refs in AI2_REFERENCE.items():
    for judge, acc in refs.items():
        if acc:
            ax.axvline(acc, ls=":", lw=1.2, label=f"{judge} ({split}, Ai2)")
ax.set_xlabel("Pairwise soft accuracy"); ax.set_xlim(0.3, 1.0); ax.invert_yaxis()
ax.legend(loc="lower right", fontsize=8)
ax.set_title("Which idea is more novel? Embedding metrics vs LLM judge")
fig.tight_layout(); fig.savefig(FIG / "pairwise_accuracy.png", dpi=150); plt.show()

In [ ]:
pt = points["Human-Only"]
kp = pt.idea.map(text_key)
first_judge = next(iter(JUDGES))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, m in zip(axes[:2], ["yin", "rnd"]):
    data = [SC.loc[kp[pt.y == y], m].values for y in (1, 0)]
    ax.boxplot(data, tick_labels=["novel", "lower-novelty"])
    ax.set_title(f"{METRICS[m]} by label (Human-Only)"); ax.set_ylabel(m)
pp = judge_point_probs(first_judge, "reference", pt)
for y, c, lab in [(1, "tab:green", "novel"), (0, "tab:red", "lower-novelty")]:
    axes[2].scatter(SC.loc[kp[pt.y == y], "yin"], pp[pt.y == y], s=12, alpha=0.6, c=c, label=lab)
axes[2].set_xlabel("Yin score"); axes[2].set_ylabel(f"{first_judge} P(novel)"); axes[2].legend(fontsize=8)
axes[2].set_title("Do the two kinds of score agree?")
fig.tight_layout(); fig.savefig(FIG / "scores_by_label.png", dpi=150); plt.show()

## Qualitative examples

Human-Only pairs where Yin and the judge (reference prompt) disagree, plus pairs both get wrong. For each idea: its Yin score and the nearest prior paper that sets it. If a "nearest paper" looks like the idea's own preprint, the leakage filter missed it.

In [ ]:
pw = pairs["Human-Only"]
ka, kb = pw.idea_a.map(text_key), pw.idea_b.map(text_key)
yin_b_wins = SC.loc[kb, "yin"].values > SC.loc[ka, "yin"].values
pb, _ = judge_pair_probs(first_judge, "reference", pw)
truth_b = pw.expected_winner.values == 1
cases = {"Yin right, judge wrong": (yin_b_wins == truth_b) & ((pb > 0.5) != truth_b),
         "Judge right, Yin wrong": (yin_b_wins != truth_b) & ((pb > 0.5) == truth_b),
         "Both wrong": (yin_b_wins != truth_b) & ((pb > 0.5) != truth_b)}
examples = []
for case, mask in cases.items():
    print(f"{case}: {mask.sum()} pairs")
    for i in np.flatnonzero(mask)[:2]:
        r = pw.iloc[i]
        for s in "ab":
            k = text_key(r[f"idea_{s}"])
            examples.append(dict(case=case, pair=int(r.id), side=s.upper(), label=r[f"label_{s}"], title=r[f"title_{s}"],
                                 rating=r[f"rating_{s}"], yin=SC.loc[k, "yin"], nearest_prior=SC.loc[k, "nn_title"],
                                 judge_p_this=pb[i] if s == "b" else 1 - pb[i]))
examples = pd.DataFrame(examples)
with pd.option_context("display.max_colwidth", 60, "display.float_format", "{:.3f}".format):
    display(examples)

## Interpretation

TODO(you): fill in after running the full notebook.

- How close do the embedding metrics get to the judge, and to Ai2's frontier judges?
- Which embedding metric wins, and does it match Liu & Zhai's ranking (Yin strongest)?
- How much does the judge move across the three prompts, compared with the embedding metrics, which can't move?
- What does the quality check say: is any score mostly tracking reviewer rating?
- On Human+Generated, does distance behave differently, and could writing style explain it?
- When would you use a distance score instead of, or alongside, an LLM judge?

## Limitations and next steps

- **Small benchmark:** 154 pairs per split, so differences of a few points are within noise. A paired bootstrap, as Ai2 used, would show which differences are real.
- **The judge is small and doesn't reason.** Reading log-probabilities with an empty thinking block understates what Qwen2.5-3B could do with reasoning, and far understates frontier judges. Compare it with Ai2's numbers as a cheap baseline, not a like-for-like rerun.
- **The corpus is one snapshot.** It ends in July 2025, two months before the deadline, and Papers with Code skews towards papers with code and benchmarks. Recent or theory-heavy prior work may be missing, which makes some ideas look more novel than they are.
- **Leakage:** the filter catches exact preprints, matching titles and near-copies, but a heavily rewritten preprint below the cosine threshold would still count as prior work against its own idea.
- **The labels mix novelty and quality** (see the confound check), and the Human+Generated labels are weak by Ai2's own description.
- **Next:** BGE-M3 embeddings (Liu & Zhai's setup), the retrieval-augmented judge (`RUN_RETRIEVAL_JUDGE`), the LFM3 contamination contrast, and Ai2's plan-format ideas (`*_plan` configs).

## Environment

In [ ]:
!pip freeze | grep -iE "^(transformers|sentence-transformers|datasets|accelerate|torch|scikit-learn|pandas|numpy|jinja2|huggingface)"

In [ ]:
ideas.drop(columns=["idea"]).to_csv(OUT / "idea_scores.csv", index=False)
leaks.to_csv(OUT / "leakage_dropped.csv", index=False)
examples.to_csv(OUT / "qualitative_examples.csv", index=False)
if IN_COLAB:
    from google.colab import files
    shutil.make_archive("novelty_results", "zip", root_dir=".", base_dir="results")
    shutil.make_archive("novelty_figures", "zip", root_dir=".", base_dir="figures")
    files.download("novelty_results.zip"); files.download("novelty_figures.zip")